# HCM AI Contest — Setup & Run
**Runtime:** A100 40GB (Colab Pro)

Chạy **5 cells** theo thứ tự. Cell cuối cùng xuất `submission/*.csv`.

In [ ]:
#@title 1. Install dependencies
#@markdown Clone repo + install requirements (~2 phút)
!git clone https://github.com/YOUR_USERNAME/aicv5.git /content/aicv5
%cd /content/aicv5
!pip install -r colab_requirements.txt -q
!pip install -q open_clip_torch

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.0f} GB")

In [ ]:
#@title 2. Mount Google Drive & link data
#@markdown Videos + keyframes + embeddings are stored on Drive.
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/aicv5')
PROJECT   = Path('/content/aicv5')

# Create symlinks so the project sees data on Drive
for name in ['videos', 'keyframes', 'persistent_store']:
    src = DRIVE_ROOT / name
    dst = PROJECT / 'data' / name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(str(src), str(dst))
        print(f"  Linked {name}")
    elif dst.exists():
        print(f"  {name} already linked")
    else:
        print(f"  WARNING: {src} not found on Drive — upload videos first!")

# Check
vid_count = len(list((PROJECT / 'data' / 'videos').glob('*.mp4'))) if (PROJECT / 'data' / 'videos').exists() else 0
print(f"\nVideos found: {vid_count}")
print(f"Drive root: {DRIVE_ROOT}")

In [ ]:
#@title 3. Index video library (keyframe extraction + embeddings)
#@markdown Skip if keyframes + persistent_store already exist on Drive.
#@markdown Takes ~30 min for 100 videos on A100.

import subprocess, sys
from pathlib import Path

PROJECT = Path('/content/aicv5')
KEYFRAMES = PROJECT / 'data' / 'keyframes'
STORE     = PROJECT / 'data' / 'persistent_store'

skip = KEYFRAMES.exists() and any(KEYFRAMES.iterdir()) and STORE.exists()
if skip:
    print("Keyframes + embeddings exist — skipping indexing.")
else:
    print("Running keyframe extraction + embedding... (this may take a while)")
    # Keyframe extraction
    sys.path.insert(0, str(PROJECT))
    from scripts.extract_keyframes import extract_all
    extract_all(
        video_dir=str(PROJECT / 'data' / 'videos'),
        output_dir=str(KEYFRAMES),
    )
    print("Keyframe extraction done.")

    # Build embeddings + persistent store
    from retrieval.persistent_store import PersistentVectorStore
    store = PersistentVectorStore(str(STORE))
    # ... populate store from keyframes + models ...
    # (connect to your actual indexing pipeline)
    print("Embeddings built.")

print(f"Keyframes: {KEYFRAMES}")
print(f"Store: {STORE}")

In [ ]:
#@title 4. Start VLM server (vLLM + Qwen2.5-VL-32B-AWQ)
#@markdown A100 40GB: runs AWQ 4-bit quantized (~20GB VRAM).
#@markdown Model download: ~30 min first time, cached on Drive.

import os, subprocess, time

DRIVE_CACHE = '/content/drive/MyDrive/aicv5/checkpoints'
os.makedirs(DRIVE_CACHE, exist_ok=True)

# Start vLLM in background — AWQ quantized for A100 40GB
VLM_CMD = (
    'python -m vllm.entrypoints.openai.api_server '
    '--model Qwen/Qwen2.5-VL-32B-Instruct-AWQ '
    '--quantization awq '
    '--max-model-len 32768 '
    '--gpu-memory-utilization 0.90 '
    '--trust-remote-code '
    '--port 8080'
)
proc = subprocess.Popen(VLM_CMD, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Wait for server to be ready
import httpx
print("Starting VLM server... ", end="")
for i in range(120):
    try:
        r = httpx.get('http://localhost:8080/health')
        if r.status_code == 200:
            print("READY")
            break
    except Exception:
        pass
    time.sleep(5)
    if i % 10 == 0:
        print(f"{i*5}s... ", end="")
else:
    print("TIMEOUT — check logs")

print(f"VLM endpoint: http://localhost:8080")
print(f"PID: {proc.pid}")

In [ ]:
#@title 5. Start API server (FastAPI + ngrok)
#@markdown Exposes the retrieval + VLM pipeline via public URL.

import subprocess, time, os
from pyngrok import ngrok

NGROK_TOKEN = ''  #@param {type:"string"}
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI in background
os.chdir('/content/aicv5')
api_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Open ngrok tunnel
time.sleep(5)
public_url = ngrok.connect(8000).public_url
print(f"API server: {public_url}")
print(f"Health check: {public_url}/api/status")

# Verify
import httpx
try:
    r = httpx.get(f'{public_url}/api/status')
    print(f"Status: {r.json().get('status', 'unknown')}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
#@title 6. Run contest — generate submission CSVs
#@markdown Upload query .txt files to Drive/aicv5/queries/, then run.

import os, sys, csv
from pathlib import Path

os.chdir('/content/aicv5')
sys.path.insert(0, '.')

from retrieval.query_parser import detect_mode, parse_trake, split_qa, TKIS, QA, TRAKE
from retrieval.frame_sampler import sample_video_windows
from retrieval.trake_aligner import align_video_events, rank_videos
from retrieval.answer_expander import expand_rows, format_csv_line, SubmissionRow
from config import settings
import httpx, json

QUERIES_DIR = '/content/drive/MyDrive/aicv5/queries'
OUTPUT_DIR  = '/content/drive/MyDrive/aicv5/submission'
API_URL     = 'http://localhost:8000'  # or ngrok public URL

os.makedirs(OUTPUT_DIR, exist_ok=True)

def search_global(query_text, top_k=20):
    """Call API for global search."""
    # Encode query via API, then search
    r = httpx.post(f'{API_URL}/api/vqa/answer', json={'question': query_text}, timeout=120)
    return r.json()

queries = sorted(Path(QUERIES_DIR).glob('*.txt'))
print(f"Found {len(queries)} query files in {QUERIES_DIR}")

for q in queries:
    mode = detect_mode(q.name)
    text = q.read_text(encoding='utf-8').strip()
    csv_path = Path(OUTPUT_DIR) / f"{q.stem}.csv"
    print(f"  [{mode.upper():5s}] {q.name} → {csv_path.name}")

    # Pipeline dispatch — connect to your full pipeline here
    # For now, write placeholder
    lines = []
    # TODO: call _tkis_pipeline / _qa_pipeline / _trake_pipeline
    # from scripts.run_contest import _tkis_pipeline, _qa_pipeline, _trake_pipeline

    with open(csv_path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')
    print(f"    → {len(lines)} rows")

print(f"\nDone. Files in {OUTPUT_DIR}/")
print("Download: right-click files → Download")